This notebook was executed on Google Colab using Google Drive for storage. The cells have been executed and outputs are preserved for review. For local testing and inference, please use the provided app.py and best.pt files in the Deployment folder."

In [1]:
# Import the drive module from the google.colab library to access cloud storage
from google.colab import drive

# Mount the Google Drive to the '/content/drive' directory to save model weights permanently
drive.mount('/content/drive')

# Install the roboflow library for dataset management and ultralytics for YOLO modeling
!pip install roboflow ultralytics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 90.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [3]:
# Import the Roboflow class from the roboflow package to connect via API
from roboflow import Roboflow

# Initialize the Roboflow client using the provided personal API authentication key
rf = Roboflow(api_key="XDLTi62LVu40JYH8GAOe")

# Access the specific workspace and project containing the annotated sperm quality dataset
project = rf.workspace("ai-driven-sperm-quality-and-selection").project("spermquality-hzctt")

# Access version 2 of the dataset project which contains the required data splits
version = project.version(2)

# Download the dataset in the PyTorch YOLO compatible format
dataset = version.download("yolo26")

# Print the local directory path where the dataset has been successfully downloaded
print(f"Dataset downloaded successfully at: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to SpermQuality-2 in yolo26:: 100%|██████████| 4812/4812 [00:01<00:00, 4795.19it/s]

Dataset downloaded successfully at: /content/SpermQuality-2


In [ ]:
# Import the YOLO class from the ultralytics library to instantiate the model
from ultralytics import YOLO

# Initialize the YOLO instance segmentation model using the medium preset weights
model = YOLO('yolo26m-seg.pt')

# Initiate the training process with specified hyperparameters to balance accuracy and memory
results = model.train(
    # Define the path to the dataset configuration YAML file
    data=f"{dataset.location}/data.yaml",

    # Set the maximum number of training epochs to 100 (early stopping is enabled by default)
    epochs=100,

    # Set the image resolution to 1024x1024 pixels to preserve microscopic morphological details
    imgsz=1024,

    # Set the batch size to 4 to prevent CUDA Out of Memory (OOM) errors during processing
    batch=4,

    # Utilize the AdamW optimizer for optimal weight decay handling in fine-grained tasks
    optimizer='AdamW',

    # Define the initial learning rate as 0.001 to ensure stable gradient descent
    lr0=0.001,

    # Specify the Google Drive path to save the training results, logs, and weights
    project='/content/drive/MyDrive/SpermQuality_Project',

    # Name the specific directory for this final training run
    name='Final_YOLO26_Run',

    # Allow overwriting of the directory if the run is restarted or executed multiple times
    exist_ok=True
)

# Print a confirmation message upon the successful completion of the training phase
print("Training Phase Complete! Model weights (best.pt) and training metrics are saved in Google Drive.")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/SpermQuality-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Final_YOLO26_Run, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pati

In [3]:
# Import the YOLO class from the ultralytics library for the evaluation phase
from ultralytics import YOLO

# Load the best performing model weights saved during the training phase from Google Drive
model = YOLO('/content/drive/MyDrive/SpermQuality_Project/Final_YOLO26_Run/weights/best.pt')

# Print a status update indicating the start of the validation evaluation
print("Initiating model evaluation on the Validation split... ")

# Run the validation process using the validation split defined in the dataset YAML
val_metrics = model.val(data=f"{dataset.location}/data.yaml", split='val')

# Print the calculated Bounding Box Mean Average Precision (mAP@50) for the validation set
print(f"Validation Set - Bounding Box mAP@50: {val_metrics.box.map50 * 100:.2f}%")

# Print the calculated Segmentation Mask Mean Average Precision (mAP@50) for the validation set
print(f"Validation Set - Segmentation Mask mAP@50: {val_metrics.seg.map50 * 100:.2f}%")

# Print a status update indicating the start of the final testing evaluation
print("\nInitiating model evaluation on the unseen Test split... ")

# Run the validation process explicitly on the test split to measure absolute generalization
test_metrics = model.val(data=f"{dataset.location}/data.yaml", split='test')

# Print the calculated Bounding Box Mean Average Precision (mAP@50) for the test set
print(f"Test Set - Bounding Box mAP@50: {test_metrics.box.map50 * 100:.2f}%")

# Print the calculated Segmentation Mask Mean Average Precision (mAP@50) for the test set
print(f"Test Set - Segmentation Mask mAP@50: {test_metrics.seg.map50 * 100:.2f}%")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Initiating model evaluation on the Validation split... 
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26m-seg summary (fused): 149 layers, 23,509,781 parameters, 0 gradients, 121.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 844.6±749.2 MB/s, size: 55.1 KB)
val: Scanning /content/SpermQuality-2/valid/labels... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 629.7it/s 0.2s
val: New cache created: /content/SpermQuality-2/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 138.1s/it 16:06
    

In [5]:
# Import YOLO for evaluation
from ultralytics import YOLO

# Load your finalized weights from the training directory
model = YOLO('/content/drive/MyDrive/SpermQuality_Project/Final_YOLO26_Run/weights/best.pt')

# Run validation with 'plots=True' to force save the Confusion Matrix
results = model.val(data=f"{dataset.location}/data.yaml", split='val', plots=True)

print("Confusion Matrix and other plots are being generated...")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26m-seg summary (fused): 149 layers, 23,509,781 parameters, 0 gradients, 121.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1311.6±454.9 MB/s, size: 53.5 KB)
val: Scanning /content/SpermQuality-2/valid/labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 3.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 147.9s/it 17:15
                   all        100       1154      0.544       0.59      0.564      0.282      0.508      0.539      0.508      0.171
                  head        100        385      0.557      0.701      0.682      0.318      0.504      0.629      0.563      0.212
                  neck        100        385      0.492      0.457      0.394      0.135       0.48      0.429      0.406      0.137
                  tail        100